# Lab 9.4 &mdash; Spans You Can Bill

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 3 &middot; Module 9 &mdash; Deployment &amp; AgentOps**

### What you'll do
- Build the span attributes from the model's own usage metadata
- Be honest about where the bill actually comes from
- Work out which questions your instrumentation can answer, and which it cannot
- Choose the labels a metric may carry, and the traces tail sampling must keep

> **How this lab works.** You write real FastAPI, Pydantic, LangChain and Kubernetes-manifest
> code. Fill every `BLANK`, then run the **Self-check** cell under each section &mdash; those
> assert on the *objects you built* (a route table, a request contract, a compiled tool, a
> manifest dict), so they are deterministic. **No graded cell needs a cluster, a running server
> or a model.** Cells marked **Run it for real** put your code in front of the sandbox model,
> your own namespace or the tracing backend; if any of those is unreachable they print how to
> fix it instead of crashing. The score line is feedback, not a grade.

> **Instrumentation is a decision made before the incident.** Every question in this
> lab is answerable or not depending on an attribute somebody chose to record weeks
> earlier, when nothing was wrong.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, math, textwrap
from typing import Any, Callable, Optional

WORK = os.path.join("/tmp", "awmas-lab-9-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model can reason before it answers, and the reasoning is billed as completion
# tokens. Off is the default here because a deployment lab makes a lot of small calls.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL, api_key=LLM_API_KEY,
                          temperature=temperature, extra_body=NO_THINK)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

# ---- your own namespace --------------------------------------------------
# You deploy into your own namespace, published at your own host. Both are injected into
# the sandbox, so nothing here is hardcoded and nothing here needs them to be set.
#
# Read ONLY from APP_NAMESPACE, never derived from the hostname. A cell below runs
# kubectl against whatever this says, and a namespace guessed from a machine name is
# the wrong thing to point kubectl at.
APP_NS   = os.environ.get("APP_NAMESPACE", "")
APP_HOST = os.environ.get("APP_HOST", "")

print("work dir :", WORK)
print("model    :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")
print("namespace:", APP_NS or "(unknown -- no graded cell needs it)")

## Concept

OpenTelemetry gives three signals and one vocabulary.

- A **trace** is one request, as a tree of **spans**. It answers *where did the time go, on this
  one*.
- A **metric** is a number over a window, cut by **labels**. It answers *how often, how bad,
  across all of them*.
- A **log** is an event with a timestamp. It answers *what exactly happened at 14:07*.

They compose: the trace ID goes in the log line, the span carries the attributes, the metric is
derived from the spans. An agent adds a fourth thing that none of the three pillars gives you
for free &mdash; **what it decided and what that cost** &mdash; and that is what this lab is about.

## Section 1 &mdash; The number that has to be on the span

Cost is a per-request property. It cannot be recovered later from a monthly invoice, and it
cannot be divided by request count &mdash; the whole point is that requests differ.

The pricing arithmetic is given. What is yours is the honest answer about where the number
you can invoice against actually comes from.

In [ ]:
from langchain_core.messages import AIMessage
from pydantic import BaseModel, Field

# USD per 1,000 tokens. Illustrative rates; the shape is what matters.
RATES = {
    "qwen-lab":  {"in": 0.0002, "out": 0.0006},
    "big-model": {"in": 0.0030, "out": 0.0150},
}


def usage_of(message: AIMessage) -> dict:
    """Token counts as LangChain normalises them, whatever the provider called its fields.

    This is where the numbers come from: the response object, not a guess and not a
    re-tokenisation of the prompt.
    """
    u = message.usage_metadata or {}
    return {"input": u.get("input_tokens", 0), "output": u.get("output_tokens", 0)}


def cost_usd(model: str, input_tokens: int, output_tokens: int) -> float:
    """What one model call cost. Rates are per 1,000 tokens and differ by direction."""
    r = RATES[model]
    return input_tokens / 1000 * r["in"] + output_tokens / 1000 * r["out"]


class SpanAttrs(BaseModel):
    """The attributes this service promises to put on every model span.

    OpenTelemetry attributes are a flat dict of strings and numbers. Writing them as a
    model is how you stop one call site quietly recording three of the five.
    """
    model_config = {"populate_by_name": True, "protected_namespaces": ()}

    model: str      = Field(alias="gen_ai.request.model")
    input_tokens: int  = Field(alias="gen_ai.usage.input_tokens")
    output_tokens: int = Field(alias="gen_ai.usage.output_tokens")
    cost_usd: float    = Field(alias="app.cost_usd")
    tenant: str        = Field(alias="app.tenant")


def span_for(message: AIMessage, model: str, tenant: str) -> dict:
    """One span's attributes, built from what the model actually returned."""
    u = usage_of(message)
    attrs = SpanAttrs(model=model, input_tokens=u["input"], output_tokens=u["output"],
                      cost_usd=cost_usd(model, u["input"], u["output"]), tenant=tenant)
    return attrs.model_dump(by_alias=True)

In [ ]:
BILL_SOURCES = (
    "the tracing UI",
    "the app.cost_usd attribute you computed and put on the span",
    "the gateway's own accounting",
)

def where_does_the_bill_come_from() -> str:
    """Which of BILL_SOURCES is the number you can actually invoice against?

    Measured on this sandbox, and this is the trap the lab is named after. The tracing
    backend's token columns read ZERO unless the observation is recorded as a GENERATION,
    and its total-token field is None regardless, because the served model has no priced
    entry in it. The trace is still worth having -- it shows the shape of the request and
    where the time went. It is not the bill.
    """
    # TODO: return one of BILL_SOURCES. The tracing UI shows what your SDK told it. Your
    # span attribute shows what YOUR rate table thinks. Only one of the three is produced
    # by the system that actually meters the tokens and charges for them.
    return BLANK

In [ ]:
# --- Self-check: Section 1   (a real AIMessage and a Pydantic model -- no model call)
def a_message() -> AIMessage:
    """Exactly what the gateway returns, minus the round trip."""
    return AIMessage(content="PMT-1003 is held pending Treasury approval.",
                     usage_metadata={"input_tokens": 900, "output_tokens": 120,
                                     "total_tokens": 1020})

def missing_attr_is_rejected() -> bool:
    """A span built without the tenant must not silently become a four-attribute span."""
    try:
        SpanAttrs(model="qwen-lab", input_tokens=900, output_tokens=120, cost_usd=0.001)
        return False
    except NameError:
        raise                 # an unfilled blank must reach check() as a NameError
    except Exception:
        return True

check("token counts come off the response object, not a guess",
      lambda: usage_of(a_message()) == {"input": 900, "output": 120})
check("a response with no usage metadata does not crash the span builder",
      lambda: usage_of(AIMessage(content="hi")) == {"input": 0, "output": 0},
      "some gateways omit it, and telemetry must never be the thing that breaks a request")
check("a thousand tokens each way on the lab model costs 0.0008",
      lambda: round(cost_usd("qwen-lab", 1000, 1000), 6) == 0.0008)
check("output tokens cost three times input on that model",
      lambda: round(cost_usd("qwen-lab", 0, 1000), 10)
              == round(3 * cost_usd("qwen-lab", 1000, 0), 10))
check("the same call on the big model costs 0.018",
      lambda: round(cost_usd("big-model", 1000, 1000), 6) == 0.018)
check("...which is 22.5x, and that ratio is a routing decision",
      lambda: round(cost_usd("big-model", 1000, 1000) / cost_usd("qwen-lab", 1000, 1000), 1)
              == 22.5)
check("the span promises all five attributes, under their OpenTelemetry names",
      lambda: set(span_for(a_message(), "qwen-lab", "ops-emea"))
              == {"gen_ai.request.model", "gen_ai.usage.input_tokens",
                  "gen_ai.usage.output_tokens", "app.cost_usd", "app.tenant"})
check("the cost is on the span, not in a log line somebody has to join",
      lambda: span_for(a_message(), "qwen-lab", "ops-emea")["app.cost_usd"] > 0)
check("...and the token counts too, so the cost can be re-derived when rates change",
      lambda: span_for(a_message(), "qwen-lab", "ops-emea")["gen_ai.usage.input_tokens"] == 900,
      "prices change; recording only the dollar figure makes the history unusable")
check("a call site that forgets an attribute fails at the schema, not in the dashboard",
      lambda: missing_attr_is_rejected())
check("THE TRACING UI IS NOT THE BILL",
      lambda: where_does_the_bill_come_from() == BILL_SOURCES[2],
      "its token columns read zero for a non-GENERATION observation, and its total is "
      "None for an unpriced model -- reconcile against the gateway's own accounting")

### Read it

`where_does_the_bill_come_from` is the point of the lab's title, so be precise about it.

The span you just built is genuinely useful: it tells you which request cost what, which tenant
is spending, and which hop is slow. What it is **not** is an invoice. Your `app.cost_usd` is your
rate table's opinion &mdash; correct only as long as somebody updates it &mdash; and the tracing
backend's own token columns are worse: they read zero unless the observation is recorded as a
`GENERATION`, and the total-token field is `None` for any model it has no price for, which
includes the one this sandbox serves.

So: instrument for **attribution and shape**, reconcile against the **gateway's accounting** for
the actual money. In this sandbox that number is on the Grafana tokenomics dashboard, not in the
tracing UI.

## Section 2 &mdash; What your instrumentation can answer

Here is one hour of a deployed service. Every request is a trace; the spans carry what somebody
decided to record &mdash; exactly the five attributes of `SpanAttrs`, plus what the HTTP layer
records for free.

In [ ]:
import random

def build_window(n: int = 240, seed: int = 9) -> list:
    """One hour of traffic. Deterministic, so everyone's numbers match."""
    rng = random.Random(seed)
    tenants = ["ops-emea", "ops-apac", "ops-us", "treasury", "client-desk"]
    out = []
    for i in range(n):
        model = "big-model" if rng.random() < 0.15 else "qwen-lab"
        pt, ct = rng.randint(600, 2400), rng.randint(80, 700)
        ok = rng.random() > 0.005
        out.append({
            "trace_id": f"{i:08x}",
            "tenant": rng.choice(tenants),
            "endpoint": rng.choice(["/ask", "/ask", "/ask", "/investigate"]),
            "payment_ref": f"PMT-{rng.randint(1000, 2200)}",
            "model": model,
            "input_tokens": pt,
            "output_tokens": ct,
            "cost_usd": cost_usd(model, pt, ct),
            "duration_s": round(rng.uniform(1.5, 12.0), 2),
            "ok": ok,
        })
    return out


WINDOW = build_window()

def spend_by(field: str) -> dict:
    """Total cost grouped by one recorded field."""
    out = {}
    for r in WINDOW:
        out[r[field]] = round(out.get(r[field], 0.0) + r["cost_usd"], 4)
    return out

In [ ]:
# What this instrumentation records -- the span attributes, plus the HTTP layer's own.
RECORDED = set(SpanAttrs.model_fields) | {"endpoint", "duration_s", "ok", "trace_id",
                                          "payment_ref"}

QUESTIONS = {
    "what did treasury spend this hour?":        {"tenant", "cost_usd"},
    "which model is the money going to?":        {"model", "cost_usd"},
    "how slow is the 95th percentile?":          {"duration_s"},
    "how often did a guardrail refuse?":         {"decision"},
    "did the answer cite a policy document?":    {"cited_policy"},
    "was the answer any good?":                  {"score"},
}

def can_answer(question: str) -> bool:
    """A question is answerable only if every attribute it needs was recorded."""
    return QUESTIONS[question] <= RECORDED

In [ ]:
# --- Self-check: Section 2
check("cost per tenant is answerable",
      lambda: can_answer("what did treasury spend this hour?"))
check("...and treasury is not the biggest spender",
      lambda: max(spend_by("tenant"), key=spend_by("tenant").get) != "treasury")
check("the model split is answerable",
      lambda: can_answer("which model is the money going to?"))
check("AN EIGHTH OF THE CALLS ARE MOST OF THE BILL",
      lambda: spend_by("model")["big-model"] > 2 * spend_by("model")["qwen-lab"],
      "a routing decision worth finding, and only visible because the model is on the span")
check("latency percentiles are answerable",
      lambda: can_answer("how slow is the 95th percentile?"))
check("but the refusal rate is NOT",
      lambda: not can_answer("how often did a guardrail refuse?"),
      "Module 8's control is invisible here -- `decision` is not in SpanAttrs")
check("nor whether the answer cited anything",
      lambda: not can_answer("did the answer cite a policy document?"),
      "Module 6's grounding check, missing from Module 9's telemetry")
check("nor whether it was any good",
      lambda: not can_answer("was the answer any good?"))
check("three of six questions cannot be answered at any price",
      lambda: sum(1 for q in QUESTIONS if not can_answer(q)) == 3,
      "not slowly, not expensively -- the data does not exist")

def _pillars():
    s = spend_by("model")
    share = s["big-model"] / sum(s.values())
    n_big = sum(1 for r in WINDOW if r["model"] == "big-model")
    print(f"  spend by model : {s}")
    print(f"  big-model      : {n_big}/{len(WINDOW)} calls, {share:.0%} of the spend")
    print(f"  spend by tenant: {spend_by('tenant')}")
    print(f"  total this hour: ${sum(s.values()):.2f}   "
          f"-> ${sum(s.values()) * 24 * 30:,.0f}/month at this rate")
    print()
    for q in QUESTIONS:
        print(f"  {'yes' if can_answer(q) else 'NO ':4} {q}")
guard(_pillars)

### Read it

The three unanswerable questions are the agent-specific ones. Latency, cost and error rate come
free with any HTTP instrumentation; *did a guardrail fire*, *did the answer cite its source* and
*was it right* have to be recorded on purpose, by you, as span attributes &mdash; which in this
notebook means adding fields to `SpanAttrs` and to every call site it validates.

That is the whole of the difference between observability for a web service and AgentOps. The
system can be perfectly healthy on all three pillars and be answering wrongly &mdash; and Module
8 closed with exactly that slide.

## Section 3 &mdash; Labels multiply, so count before you add one

`payment_ref` is on the span and that is correct. Putting it on a **metric** is a different act
with a different cost, because every distinct value creates a time series that is stored,
indexed and queried forever.

In [ ]:
CARDINALITY = {"service": 1, "endpoint": 4, "status": 3, "model": 2,
               "tenant": 5, "payment_ref": 1200}
SERIES_BUDGET = 500

def series_count(labels) -> int:
    """How many time series does one metric with these labels produce?

    Labels do not add. Each one MULTIPLIES the series count by its distinct values.
    """
    return math.prod(CARDINALITY[l] for l in labels)


def safe_to_label(labels, budget: int = SERIES_BUDGET) -> bool:
    return series_count(labels) <= budget


def label_set_for_metric() -> list:
    """The labels you are willing to put on `agent_requests_total`.

    Every one of them is also on the span, where it costs nothing extra. Putting it on a
    METRIC is a different act: a time series per distinct combination, stored and indexed
    forever.
    """
    # TODO: choose from CARDINALITY. You want to be able to cut this metric by endpoint,
    # status, model and tenant -- and you have a budget of SERIES_BUDGET series for it.
    # Price your answer with series_count() before you commit to it.
    return BLANK

In [ ]:
# --- Self-check: Section 3
BASE = ["service", "endpoint", "status"]

check("the base label set is twelve series",
      lambda: series_count(BASE) == 12)
check("adding the model doubles it",
      lambda: series_count(BASE + ["model"]) == 24)
check("adding the tenant is still fine",
      lambda: series_count(BASE + ["model", "tenant"]) == 120)
check("ADDING THE PAYMENT REFERENCE IS 144,000 SERIES",
      lambda: series_count(BASE + ["model", "tenant", "payment_ref"]) == 144000,
      "for one metric -- and payment_ref is unbounded, so that number only grows")
check("your label set stays inside the budget",
      lambda: safe_to_label(label_set_for_metric()),
      "price it with series_count() before you commit -- the budget is SERIES_BUDGET")
check("...and still lets you cut the metric four ways",
      lambda: {"endpoint", "status", "model", "tenant"} <= set(label_set_for_metric()))
check("...without the payment reference",
      lambda: "payment_ref" not in label_set_for_metric(),
      "one unbounded label is how a metrics bill triples in a fortnight")
check("the same field on a SPAN costs nothing extra",
      lambda: "payment_ref" in RECORDED,
      "spans are stored per request; labels are stored per distinct combination, forever")

def _labels():
    for extra in ([], ["model"], ["model", "tenant"], ["model", "tenant", "payment_ref"]):
        ls = BASE + extra
        print(f"  {series_count(ls):>7,} series  {'ok ' if safe_to_label(ls) else 'NO '}"
              f" {'+'.join(ls)}")
    guard(lambda: print(f"\n  your choice: {series_count(label_set_for_metric()):,} series"))
guard(_labels)

## Section 4 &mdash; The trace you need is the one you did not keep

Traces are the expensive signal, so everybody samples. Head sampling &mdash; decide at the start
of the request, keep 10% &mdash; is the default because it is the cheapest to implement.

Do the arithmetic on it once and you will not use it for an agent service. The arithmetic is
given; the **keep rule** is yours.

In [ ]:
DAILY_REQUESTS = 2000
FAILURE_RATE   = 1 / 200          # the thing you will be asked about
SLOW_S         = 11.5             # roughly the slowest 5% of requests

def expected_captured(p: float, requests: int = DAILY_REQUESTS,
                      failure_rate: float = FAILURE_RATE) -> float:
    """How many of the day's failures head sampling at rate p keeps."""
    return requests * failure_rate * p


def p_miss_everything(p: float, requests: int = DAILY_REQUESTS,
                      failure_rate: float = FAILURE_RATE) -> float:
    """The chance that a whole day of head sampling keeps NOT ONE failing trace.

    Each failure is kept independently with probability p, so all of them are dropped
    with probability (1 - p) raised to the number of failures.
    """
    return (1 - p) ** (requests * failure_rate)


def keep_trace(trace: dict) -> bool:
    """Tail sampling: decide when the request ENDS, with the outcome in hand.

    Keeping 5% of ordinary traffic is a cost decision. The question is which traces are
    kept regardless of that 5%.
    """
    if not trace["ok"]:
        return True                                  # errors, always
    if trace["duration_s"] > SLOW_S:
        return True                                  # the slow tail, always
    # TODO: two agent outcomes get asked about in every incident review, and neither of
    # them is an error -- the service returned 200 and did exactly the right thing. Which
    # two values of `decision` must always be kept? (the third is the ordinary one)
    if trace["decision"] in BLANK:
        return True
    return trace["sample_roll"] < 0.05

In [ ]:
def sample_window(n: int = 500, seed: int = 4) -> list:
    """A window of FINISHED traces: outcome known, which is what tail sampling gets to see
    and head sampling does not. `decision` is here because Your-turn item 1 put it on the
    span -- a keep rule can only branch on attributes somebody recorded."""
    rng = random.Random(seed)
    out = []
    for _ in range(n):
        r = rng.random()
        out.append({"ok": rng.random() > FAILURE_RATE,
                    "duration_s": round(rng.uniform(1.5, 12.0), 2),
                    "decision": ("refused" if r < 0.04
                                 else "escalated" if r < 0.07 else "answered"),
                    "sample_roll": rng.random()})
    return out


SAMPLE_TRACES = sample_window()

def tail_kept_fraction(traces=None) -> float:
    """What fraction of a window tail sampling actually stores, under YOUR keep rule."""
    rows = traces if traces is not None else SAMPLE_TRACES
    return sum(1 for t in rows if keep_trace(t)) / len(rows)

In [ ]:
# --- Self-check: Section 4
def _of(**over):
    """One trace, mostly ordinary, overridden where the check cares."""
    base = {"ok": True, "duration_s": 3.0, "decision": "answered", "sample_roll": 0.99}
    return {**base, **over}

check("there are ten failures in a day at this rate",
      lambda: DAILY_REQUESTS * FAILURE_RATE == 10)
check("head sampling at 10% expects to keep exactly one of them",
      lambda: expected_captured(0.10) == 1.0)
check("...and on 35% of days it keeps none at all",
      lambda: round(p_miss_everything(0.10), 4) == 0.3487,
      "one day in three, the trace the incident review asks for was never stored")
check("even 50% head sampling loses every failure on 1 day in 1000",
      lambda: round(p_miss_everything(0.50), 4) == 0.001)
check("your keep rule keeps every error",
      lambda: keep_trace(_of(ok=False)) is True)
check("...and every slow request",
      lambda: keep_trace(_of(duration_s=SLOW_S + 1)) is True)
check("A REFUSAL IS KEPT, THOUGH IT IS A 200 AND NOTHING WENT WRONG",
      lambda: keep_trace(_of(decision="refused")) is True,
      "it is the first thing anyone asks about, and it is not an error")
check("an escalation is kept for the same reason",
      lambda: keep_trace(_of(decision="escalated")) is True)
check("an ordinary fast successful answer is sampled, not kept",
      lambda: keep_trace(_of(sample_roll=0.99)) is False)
check("...and one in twenty of those is kept anyway",
      lambda: keep_trace(_of(sample_roll=0.01)) is True)
check("the rule stores well under a fifth of the window",
      lambda: tail_kept_fraction() < 0.20,
      "and unlike head sampling at 10%, it has lost none of the interesting ones")

def _sampling():
    print(f"  {'strategy':30} {'stored':>8} {'failures kept':>14} {'blind days':>11}")
    for p in (0.01, 0.10, 0.50):
        print(f"  head sampling at {p:>4.0%}           {p:>7.1%} "
              f"{expected_captured(p):>13.1f} {p_miss_everything(p):>10.1%}")
    guard(lambda: print(f"  tail sampling, your keep rule  {tail_kept_fraction():>7.1%} "
                        f"{'all':>13} {0.0:>10.1%}"))
guard(_sampling)

### Read it

Head sampling at 10% is the industry default and it loses the entire day's evidence about one
day in three. That is not a tail risk, it is a coin you flip every incident review.

Tail sampling costs more to run &mdash; the collector must buffer each trace until the request
finishes, which is why this decision belongs in the **collector** and not in your application.
That is the practical reason the OTLP collector exists between your process and your backend: it
is the one place where sampling, redaction and fan-out to several destinations can happen
without a redeploy of the service.

And note what your keep rule had to know: `decision`. A keep rule can only branch on attributes
somebody recorded &mdash; which is Section 2 again, from the other end.

## Run it for real

Send one trace to the tracing backend, with the cost attributes on it. Your sandbox is pointed
at a shared project and separated by environment, so you will see your own traces and not
anybody else's.

In [ ]:
def send_trace():
    host = os.environ.get("LANGFUSE_HOST") or os.environ.get("LANGFUSE_BASE_URL")
    if not (host and os.environ.get("LANGFUSE_PUBLIC_KEY")
            and os.environ.get("LANGFUSE_SECRET_KEY")):
        print("Tracing is not configured here. To point at a backend, set LANGFUSE_HOST,")
        print("LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY. Nothing above needed them.")
        return
    if "us.cloud.langfuse.com" in host:
        print("That host accepts the export with an HTTP 200 and never renders the trace.")
        print("Point LANGFUSE_HOST at the region your keys belong to and try again -- keys")
        print("are region-bound, so a different region also means a different key pair.")
        return

    from langfuse import Langfuse
    lf = Langfuse(host=host)            # keys come from the environment
    if not lf.auth_check():
        print("Credentials rejected. Keys are region-bound -- check the host matches them.")
        return

    def usage(i, o):
        # total_tokens is REQUIRED by langchain-core's UsageMetadata -- omitting it raises
        # a ValidationError, not a warning.
        return {"input_tokens": i, "output_tokens": o, "total_tokens": i + o}

    msgs = [AIMessage(content="plan",     usage_metadata=usage(900, 120)),
            AIMessage(content="retrieve", usage_metadata=usage(240, 40)),
            AIMessage(content="answer",   usage_metadata=usage(2100, 480))]
    steps = [("plan", "qwen-lab"), ("retrieve", "qwen-lab"), ("answer", "big-model")]
    spans = [span_for(m, model, "ops-emea") for m, (_, model) in zip(msgs, steps)]

    # SDK 4.x: observations nest by being entered inside one another. client.trace(...)
    # is the v3 API and does not exist here.
    with lf.start_as_current_observation(name="investigate-payment", as_type="span") as root:
        root.update(metadata={"app.cost_usd": round(sum(s["app.cost_usd"] for s in spans), 6),
                              "app.tenant": "ops-emea", "payment_ref": "PMT-1003"})
        for (name, _), attrs in zip(steps, spans):
            with lf.start_as_current_observation(name=name, as_type="span") as obs:
                obs.update(metadata=attrs)
    lf.flush()

    env = os.environ.get("LANGFUSE_TRACING_ENVIRONMENT", "(unset)")
    print(f"sent one trace to {host}")
    print(f"environment tag: {env}  -- filter on it in the UI to see only your own")
    print("Open it and check two things: the cost is on the root as well as the leaves,")
    print("and the token COLUMNS read zero. These are spans, not generations -- which is")
    print("exactly why Section 1's answer is what it is.")

guard(send_trace)

### Read it

Note what separated your traces from everyone else's: an environment variable, read by the SDK,
with no code of yours involved. That is worth copying &mdash; per-tenant or per-environment
separation that depends on every call site remembering to pass a field is separation that lasts
until the first new call site.

Note also what the UI shows in the token columns: zero. Your attributes are all there in the
metadata, and the backend's own accounting of them is empty, because a span is not a generation
and the served model has no price entry. The trace is the shape of the request. The bill comes
from the gateway.

The same trace, exported through an OTLP collector to any other backend, would carry the same
attributes under the same `gen_ai.*` names. That naming convention is why the choice of backend
is reversible and the choice of *what to record* is not.

In [ ]:
score()

## Your turn

1. Add `decision` and `cited_policy` to `SpanAttrs` and re-run Section 2. Two questions become
   answerable; work out what it would have cost to add them after the incident instead of before.
2. `tenant` is a span attribute and a metric label. Decide what each one is for &mdash; you
   probably want both &mdash; and write down which question you would answer from which.
3. Extend the keep rule with one more class: traces whose `app.cost_usd` is in the top 1%. Then
   re-run `tail_kept_fraction()` and check you can still afford it.